# 03 — The IQ ↔ audio bridge (SigMF)

**Thesis:** a radio IQ capture is the *same DNA* as audio PCM — a raw stream of
interleaved integer samples with metadata beside it. Here we dissect a real
SigMF recording (Tianwen-2's X-band downlink, caught by the Dwingeloo dish) with
the same tools we'd point at a WAV, and sketch the acidcat walker it deserves.

Datatype `ci16_le` per the SigMF spec = **c**omplex **i**nterleaved signed
**16**-bit, **l**ittle-**e**ndian, I first: `I0 Q0 I1 Q1 …` (4 bytes / IQ sample).

In [ ]:
import numpy as np, json, matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path

# the capture (point elsewhere as more land in specimens/library/sdr-captures/)
BASE = Path('~/lab/notebooks/Satellite-Hacking/Tianwen/tianwen2').expanduser()
STEM = 'dwingeloo_kamo_2026_05_25_19_43_00_8428.190MHz_2.00Msps_ci16_le'
DATA = BASE / (STEM + '.sigmf-data')
META = BASE / (STEM + '.sigmf-meta')
print('data:', DATA.stat().st_size // 1_000_000, 'MB   meta:', META.stat().st_size, 'B')

## 1. The SigMF sidecar — metadata beside the data

JSON, exactly like a WAV's `bext`/`iXML` but richer (this one carries telescope pointing).

In [ ]:
meta = json.loads(META.read_text())
g = meta['global']; cap = meta['captures'][0]
fs = int(g['core:sample_rate']); dtype = g['core:datatype']
fc = int(cap['core:frequency'])
print(f"datatype   {dtype}")
print(f"sample_rate {fs/1e6:g} Msps")
print(f"center freq {fc/1e6:.3f} MHz  (maps to the DC bin)")
print(f"recorder   {g.get('core:recorder')}  |  {g.get('core:description')}")
print(f"rx_gain    {g.get('vrt:rx_gain')} dB   bandwidth {g.get('vrt:bandwidth',0)/1e6:g} MHz")
print(f"pointing   az {g.get('dt:pointing:current:az_deg')}°  el {g.get('dt:pointing:current:el_deg')}°")

## 2. The raw bytes *are* interleaved PCM

Read a slice as `int16` and split into I/Q — identical to splitting stereo L/R.

In [ ]:
# first 8 int16s = the first 4 complex samples, no header
head = np.fromfile(DATA, dtype='<i2', count=8)
print('first int16s:', head.tolist())
print('as I/Q pairs:', list(zip(head[0::2], head[1::2])))

# read ~2 s of capture: N complex samples = 2N int16
N = 4_000_000
raw = np.fromfile(DATA, dtype='<i2', count=2*N).astype(np.float32)
iq = (raw[0::2] + 1j*raw[1::2]) / 32768.0   # scale int16 -> ~[-1,1), optional
print(f'{len(iq):,} complex samples  ({len(iq)/fs:.2f} s)')

## 3. Time domain — I and Q

In [ ]:
w = slice(0, 1500)
t = np.arange(w.stop) / fs * 1e3
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(t, iq[w].real, lw=.7, label='I'); ax[0].plot(t, iq[w].imag, lw=.7, label='Q')
ax[0].legend(loc='upper right'); ax[0].set(title='I / Q (in-phase & quadrature)', ylabel='amplitude')
ax[1].plot(t, np.abs(iq[w]), lw=.7, color='crimson')
ax[1].set(title='magnitude |I+jQ|', xlabel='time (ms)', ylabel='|amp|')
plt.tight_layout(); plt.show()

## 4. The spectrum — RF-correct (two-sided, centered on the carrier)

For a **complex** signal the FFT is two-sided (negative and positive frequencies
both real), so the band spans `center ± fs/2`. Welch PSD + `fftshift`, absolute axis.

In [ ]:
f, Pxx = signal.welch(iq, fs=fs, nperseg=8192, return_onesided=False, detrend=False)
f = np.fft.fftshift(f); Pxx = np.fft.fftshift(Pxx)
freq_mhz = (f + fc) / 1e6
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(freq_mhz, 10*np.log10(Pxx + 1e-20), lw=.8)
ax.axvline(fc/1e6, color='crimson', ls='--', lw=.8, label=f'carrier {fc/1e6:.3f} MHz')
ax.set(title='power spectral density', xlabel='frequency (MHz, absolute RF)', ylabel='dB/Hz')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Waterfall — the signal over time

The classic SDR view: a spectrogram of the *complex* signal (two-sided, shifted).

In [ ]:
f, tt, Sxx = signal.spectrogram(iq, fs=fs, nperseg=2048, noverlap=1024,
                                return_onesided=False, detrend=False, mode='psd')
f = np.fft.fftshift(f); Sxx = np.fft.fftshift(Sxx, axes=0)
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(10*np.log10(Sxx + 1e-20), aspect='auto', origin='lower', cmap='magma',
               extent=[tt[0], tt[-1], (f[0]+fc)/1e6, (f[-1]+fc)/1e6])
ax.set(title='waterfall (Tianwen-2 X-band downlink)', xlabel='time (s)', ylabel='frequency (MHz)')
fig.colorbar(im, ax=ax, label='dB'); plt.tight_layout(); plt.show()

## 6. Same DSP as audio

Everything above — read interleaved integers, FFT, spectrogram — is *identical* to
audio analysis. The only differences: audio samples are **real** (one number/sample,
one-sided spectrum) while IQ is **complex** (two numbers, two-sided); and IQ carries
a **center frequency** that offsets the whole band up to RF. Rename this `.sigmf-data`
to `.raw`, import into Audacity as *signed 16-bit LE, 2 channels*, and it loads.

## 7. Prototype: the acidcat walker this format deserves

acidcat's contract is `walk(path) -> (fmt_label, chunks, file_warnings)`. SigMF maps
cleanly: the JSON sidecar's `global` + each `capture` become field-bearing 'chunks',
and the datatype string tells us the sample geometry. Seed for the dev proposal.

In [ ]:
def sigmf_walk(data_path):
    data_path = Path(data_path)
    meta_path = data_path.with_suffix('.sigmf-meta')
    warns = []
    if not meta_path.exists():
        warns.append('no .sigmf-meta sidecar; datatype unknown (SigMF requires the pair)')
        return 'SigMF/IQ (headerless)', [], warns
    m = json.loads(meta_path.read_text()); g = m['global']
    dt = g.get('core:datatype', '')
    # parse ci16_le -> complex?, bits, signed?, endian
    cplx = dt.startswith('c'); body = dt[1:] if dt[:1] in 'cr' else dt
    kind = {'i':'int','u':'uint','f':'float'}.get(body[:1], '?')
    bits = int(''.join(ch for ch in body if ch.isdigit()) or 0)
    comp_bytes = bits // 8; samp_bytes = comp_bytes * (2 if cplx else 1)
    n_samp = data_path.stat().st_size // samp_bytes if samp_bytes else 0
    fs = g.get('core:sample_rate')
    chunks = [
        {'id': 'global', 'offset': 0, 'size': len(json.dumps(g)),
         'summary': f"{dt}  {(fs or 0)/1e6:g} Msps  {n_samp:,} samples",
         'fields': [
            {'off': 0, 'len': 0, 'name': 'datatype', 'value': dt,
             'note': f'{"complex" if cplx else "real"} {kind}{bits}, {samp_bytes} B/sample'},
            {'off': 0, 'len': 0, 'name': 'sample_rate', 'value': fs, 'note': 'Hz'},
            {'off': 0, 'len': 0, 'name': 'sample_count', 'value': n_samp, 'note': 'derived from file size'},
         ], 'warnings': []},
    ]
    for i, c in enumerate(m.get('captures', [])):
        chunks.append({'id': f'capture[{i}]', 'offset': c.get('core:sample_start', 0), 'size': 0,
            'summary': f"@ {(c.get('core:frequency') or 0)/1e6:.3f} MHz",
            'fields': [{'off': 0, 'len': 0, 'name': k.split(':')[-1], 'value': v, 'note': ''}
                       for k, v in c.items()], 'warnings': []})
    if fs and (data_path.stat().st_size % samp_bytes):
        warns.append('file size not a whole number of samples')
    return 'SigMF/IQ', chunks, warns

fmt, chunks, warns = sigmf_walk(DATA)
print('format:', fmt, '| warnings:', warns or 'none')
for c in chunks:
    print(f"  [{c['id']}] {c['summary']}")
    for fld in c['fields'][:4]:
        print(f"      {fld['name']}: {fld['value']}  {('('+fld['note']+')') if fld['note'] else ''}")

---
**Dev proposal:** promote `sigmf_walk` to `core/walk/sigmf.py` behind the 3-step
extension point (sniff the `.sigmf-meta` JSON / `SigMF` magic, return this shape,
one `_WALKERS` entry). Then extend to PortaPack `.C16`+`.TXT`, RTL `.cu8`, and
IQ-in-WAV — the whole SDR family, in the tool that already speaks audio containers.